# [Ensemble] Nodewise Closed-Form (Event-Balanced)

This notebook builds a nodewise closed-form ensemble from model bundles and writes a full `submission.csv`.

It keeps the same overall workflow as the standard nodewise closed-form notebook:

- detect compatible bundles
- load OOF predictions target by target
- solve simplex-constrained nodewise blend weights
- build a final submission with robust merge fallbacks

The difference is the objective used when solving nodewise weights.

Instead of optimizing ordinary row-wise SSE, this notebook uses an **event-balanced weighted SSE** objective.
Each OOF row is weighted by `1 / T_event`, where `T_event` is the number of rows for that node and event, so longer events do not dominate the closed-form solve.

This is intended to move the learned weights closer to the competition-style event-average evaluation, while still keeping a fast closed-form solution.
It is still an approximation to event-average RMSE, not an exact optimization of the leaderboard metric.

CV reporting remains event-average standardized RMSE, consistent with the current ensemble notebooks.

Outputs:

- `submission.csv`
- `node_weights.csv`
- `cv_summary.csv`
- `detected_sources.csv`
- `blend_metadata.json`
- optional `oof_blend_predictions.parquet/csv`


In [1]:
import os
import gc
import json
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Iterable, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd

# ============================================================
# Config
# ============================================================

SCHEMA_VERSION = "ufb_ensemble_v1_nodewise_eventbalanced"
OOF_KEY_COLS = ["model_id", "event_id", "node_type", "node_id", "timestep"]
NODE_GROUP_COLS = ["model_id", "node_type", "node_id"]
NODE_KEY_COLS = ["model_id", "event_id", "node_type", "node_id"]
OOF_REQ_COLS = OOF_KEY_COLS + ["water_level_true", "water_level_pred"]
TEST_REQ_MIN_COLS = NODE_KEY_COLS + ["water_level_pred"]

STD_DEV = {
    (1, 1): 16.877747,
    (1, 2): 14.378797,
    (2, 1): 3.191784,
    (2, 2): 2.727131,
}


def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, str(default)))


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def parse_path_list(raw: str) -> List[Path]:
    out = []
    for x in raw.split(","):
        x = x.strip()
        if not x:
            continue
        out.append(Path(x))
    return out


def parse_targets(raw: str) -> List[Tuple[int, int]]:
    out: List[Tuple[int, int]] = []
    for tok in raw.split(","):
        tok = tok.strip()
        if not tok:
            continue
        parts = tok.split(":")
        if len(parts) != 2:
            raise ValueError(f"invalid TARGETS token: {tok}")
        out.append((int(parts[0]), int(parts[1])))
    if not out:
        raise ValueError("TARGETS is empty")
    return out


BUNDLE_ROOTS = parse_path_list(
    os.environ.get(
        "BUNDLE_ROOTS",
        "/kaggle/input,/kaggle/working/model_bundles,ensemble/artifacts/model_bundles",
    )
)
MANIFEST_NAME = os.environ.get("MANIFEST_NAME", "bundle_manifest.json")
STRICT_REQUIRE_MANIFESTS = env_bool("STRICT_REQUIRE_MANIFESTS", False)

EXPECTED_FOLDS = env_int("EXPECTED_FOLDS", 5)
EXPECTED_SEED = env_int("EXPECTED_SEED", 2026)
ALLOW_FOLD_MISMATCH = env_bool("ALLOW_FOLD_MISMATCH", True)

MAX_MODELS_EXACT = env_int("MAX_MODELS_EXACT", 8)
MIN_OOF_ROWS_NODE = env_int("MIN_OOF_ROWS_NODE", 64)
LOG_EVERY_NODES = env_int("LOG_EVERY_NODES", 200)
SAVE_BLEND_OOF = env_bool("SAVE_BLEND_OOF", True)
SUBMISSION_BLEND_CHUNK_ROWS = max(100_000, env_int("SUBMISSION_BLEND_CHUNK_ROWS", 2_000_000))
SUBMISSION_LOG_EVERY_CHUNKS = max(1, env_int("SUBMISSION_LOG_EVERY_CHUNKS", 10))

# Memory/speed controls
DEFER_TEST_LOAD = env_bool("DEFER_TEST_LOAD", True)
SORT_NORMALIZED_TABLES = env_bool("SORT_NORMALIZED_TABLES", False)
STRICT_TABLE_CHECKS = env_bool("STRICT_TABLE_CHECKS", False)
COMPUTE_SOURCE_NODE_GROUPS = env_bool("COMPUTE_SOURCE_NODE_GROUPS", False)
OOF_FILTER_FINITE = env_bool("OOF_FILTER_FINITE", True)
TEST_FILTER_FINITE = env_bool("TEST_FILTER_FINITE", True)

TARGETS = parse_targets(os.environ.get("TARGETS", "1:1,1:2,2:1,2:2"))
OOF_CSV_CHUNK_ROWS = max(200_000, env_int("OOF_CSV_CHUNK_ROWS", 2_000_000))
OOF_PARQUET_BATCH_ROWS = max(200_000, env_int("OOF_PARQUET_BATCH_ROWS", 2_000_000))
INCLUDE_OOF_EMPTY_SOURCES_IN_TEST_BLEND = env_bool("INCLUDE_OOF_EMPTY_SOURCES_IN_TEST_BLEND", False)
TEST_CSV_CHUNK_ROWS = max(200_000, env_int("TEST_CSV_CHUNK_ROWS", 2_000_000))
TEST_PARQUET_BATCH_ROWS = max(200_000, env_int("TEST_PARQUET_BATCH_ROWS", 2_000_000))
TEST_STREAM_ROWID_FASTPATH = env_bool("TEST_STREAM_ROWID_FASTPATH", True)
TEST_FILTER_BY_SOURCE_TARGETS = env_bool("TEST_FILTER_BY_SOURCE_TARGETS", True)
TEST_SOURCE_PARALLEL = max(1, env_int("TEST_SOURCE_PARALLEL", 4))

SAMPLE_SUBMISSION_PATH = os.environ.get("SAMPLE_SUBMISSION_PATH", "")
DEFAULT_OUTPUT_DIR = (
    "/kaggle/working/ensemble_output_auto_nodewise_eventbalanced"
    if Path("/kaggle/working").exists()
    else "ensemble/artifacts/ensemble_runs/auto_nodewise_eventbalanced"
)
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", DEFAULT_OUTPUT_DIR))
PUBLIC_SUBMISSION_PATH = os.environ.get(
    "PUBLIC_SUBMISSION_PATH",
    "/kaggle/working/submission.csv" if Path("/kaggle/working").exists() else "",
).strip()


# ============================================================
# Logging
# ============================================================

def log(msg: str) -> None:
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)


# ============================================================
# Bundle discovery
# ============================================================

@dataclass
class Bundle:
    name: str
    bundle_name: str
    source_kind: str
    n_folds: int
    event_split_seed: int
    oof_path: Path
    test_path: Path
    manifest_path: Path | None = None



def _dtype_map_for_columns(cols: List[str]) -> Dict[str, str]:
    base = {
        "row_id": "int64",
        "model_id": "int16",
        "event_id": "int32",
        "node_type": "int16",
        "node_id": "int32",
        "timestep": "int32",
        "water_level_true": "float32",
        "water_level_pred": "float32",
    }
    return {c: t for c, t in base.items() if c in cols}


def read_table(path: Path, usecols: List[str] | None = None, kind: str = "generic") -> pd.DataFrame:
    suffix = path.suffix.lower()
    if usecols is None:
        if kind == "oof":
            usecols = list(OOF_REQ_COLS)
        elif kind == "test":
            usecols = [
                c
                for c in ["row_id", "model_id", "event_id", "node_type", "node_id", "timestep", "water_level_pred"]
            ]

    if suffix == ".csv":
        header = pd.read_csv(path, nrows=0)
        cols = header.columns.tolist()
        if usecols is not None:
            cols = [c for c in usecols if c in cols]
        dtypes = _dtype_map_for_columns(cols)
        return pd.read_csv(path, usecols=cols, dtype=dtypes if dtypes else None, low_memory=False)

    if suffix in {".parquet", ".pq"}:
        cols = None if usecols is None else list(usecols)
        if cols is not None:
            try:
                import pyarrow.parquet as pq
                schema_cols = set(pq.ParquetFile(path).schema.names)
                cols = [c for c in cols if c in schema_cols]
            except Exception:
                pass
        df = pd.read_parquet(path, columns=cols)
        dtypes = _dtype_map_for_columns(df.columns.tolist())
        for c, dt in dtypes.items():
            if c in df.columns:
                try:
                    df[c] = df[c].astype(dt, copy=False)
                except Exception:
                    pass
        return df

    raise ValueError(f"unsupported table format: {path}")



def discover_manifest_paths(roots: Iterable[Path], manifest_name: str) -> List[Path]:
    out: List[Path] = []
    for root in roots:
        if not root.exists():
            continue
        out.extend(sorted(root.rglob(manifest_name)))
    return out



def _required(raw: Dict, key: str):
    if key not in raw:
        raise KeyError(f"manifest missing required key: {key}")
    return raw[key]



def _resolve_rel(manifest_path: Path, rel_or_abs: str) -> Path:
    p = Path(rel_or_abs)
    if p.is_absolute():
        return p
    return (manifest_path.parent / p).resolve()



def load_manifest_bundle(manifest_path: Path) -> Bundle:
    raw = json.loads(manifest_path.read_text())
    schema = str(_required(raw, "schema_version"))
    if schema != SCHEMA_VERSION:
        raise ValueError(f"{manifest_path}: schema_version mismatch: {schema}")

    name = str(_required(raw, "bundle_name")).strip()
    if not name:
        raise ValueError(f"{manifest_path}: empty bundle_name")

    n_folds = int(_required(raw, "n_folds"))
    seed = int(_required(raw, "event_split_seed"))
    oof_path = _resolve_rel(manifest_path, str(_required(raw, "oof_predictions_path")))
    test_path = _resolve_rel(manifest_path, str(_required(raw, "test_predictions_path")))

    if not oof_path.exists():
        raise FileNotFoundError(f"{manifest_path}: missing oof file: {oof_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"{manifest_path}: missing test file: {test_path}")

    return Bundle(
        name=name,
        bundle_name=name,
        source_kind="manifest",
        n_folds=n_folds,
        event_split_seed=seed,
        oof_path=oof_path,
        test_path=test_path,
        manifest_path=manifest_path,
    )



def discover_loose_bundles(roots: Iterable[Path]) -> List[Bundle]:
    """Fallback discovery when manifest is missing.

    Looks for dirs containing both an OOF-like and test-like table.
    """
    dir_map: Dict[Path, Dict[str, List[Path]]] = {}

    def mark(parent: Path, kind: str, path: Path) -> None:
        rec = dir_map.setdefault(parent, {"oof": [], "test": []})
        rec[kind].append(path)

    table_patterns = ["*.csv", "*.parquet", "*.pq"]
    for root in roots:
        if not root.exists():
            continue
        for pat in table_patterns:
            for p in root.rglob(pat):
                lower = p.name.lower()
                if "oof" in lower and ("pred" in lower or "prediction" in lower):
                    mark(p.parent, "oof", p)
                elif "test" in lower and ("pred" in lower or "prediction" in lower):
                    mark(p.parent, "test", p)

    out: List[Bundle] = []
    used_names = set()
    for d, rec in sorted(dir_map.items(), key=lambda x: str(x[0])):
        if not rec["oof"] or not rec["test"]:
            continue
        oof_path = sorted(rec["oof"])[0]
        test_path = sorted(rec["test"])[0]
        raw_name = f"loose__{d.name}"
        name = raw_name
        i = 2
        while name in used_names:
            name = f"{raw_name}_{i}"
            i += 1
        used_names.add(name)

        out.append(
            Bundle(
                name=name,
                bundle_name=raw_name,
                source_kind="loose",
                n_folds=-1,
                event_split_seed=-1,
                oof_path=oof_path,
                test_path=test_path,
                manifest_path=None,
            )
        )
    return out


def uniquify_bundle_names(bundles: List[Bundle]) -> List[Bundle]:
    out: List[Bundle] = []
    used = set()
    seen_by_bundle_name: Dict[str, int] = {}
    for b in bundles:
        base = str(b.bundle_name).strip() or str(b.name).strip()
        count = seen_by_bundle_name.get(base, 0) + 1
        seen_by_bundle_name[base] = count
        name = base if count == 1 else f"{base}__dup{count}"
        while name in used:
            count += 1
            seen_by_bundle_name[base] = count
            name = f"{base}__dup{count}"
        if name != b.name:
            log(f"duplicate bundle name aliased: {base} -> {name} ({b.manifest_path})")
        b.name = name
        used.add(name)
        out.append(b)
    return out


# ============================================================
# Normalize tables
# ============================================================


def normalize_oof(df: pd.DataFrame, source: str) -> pd.DataFrame:
    miss = [c for c in OOF_REQ_COLS if c not in df.columns]
    if miss:
        raise KeyError(f"{source}: OOF missing columns: {miss}")

    out = df[OOF_REQ_COLS]

    if OOF_FILTER_FINITE:
        a_true = out["water_level_true"].to_numpy(dtype=np.float32, copy=False)
        a_pred = out["water_level_pred"].to_numpy(dtype=np.float32, copy=False)
        keep = np.isfinite(a_true) & np.isfinite(a_pred)
        if not np.all(keep):
            out = out.loc[keep]

    if SORT_NORMALIZED_TABLES:
        out = out.sort_values(OOF_KEY_COLS)

    if STRICT_TABLE_CHECKS:
        if out.duplicated(OOF_KEY_COLS).any():
            raise ValueError(f"{source}: duplicated OOF keys")

    return out.reset_index(drop=True)


def normalize_test(df: pd.DataFrame, source: str) -> pd.DataFrame:
    miss = [c for c in TEST_REQ_MIN_COLS if c not in df.columns]
    if miss:
        raise KeyError(f"{source}: test missing columns: {miss}")

    keep = [c for c in ["row_id", "model_id", "event_id", "node_type", "node_id", "timestep", "water_level_pred"] if c in df.columns]
    out = df[keep]

    if TEST_FILTER_FINITE and "water_level_pred" in out.columns:
        pred = out["water_level_pred"].to_numpy(dtype=np.float32, copy=False)
        keep_mask = np.isfinite(pred)
        if not np.all(keep_mask):
            out = out.loc[keep_mask]

    if SORT_NORMALIZED_TABLES:
        if "row_id" in out.columns:
            out = out.sort_values(["row_id"])
        else:
            sort_cols = [c for c in ["model_id", "event_id", "node_type", "node_id", "timestep"] if c in out.columns]
            out = out.sort_values(sort_cols)

    if STRICT_TABLE_CHECKS and "row_id" in out.columns:
        if out.duplicated(["row_id"]).any():
            raise ValueError(f"{source}: duplicated row_id in test")

    return out.reset_index(drop=True)


def normalize_test_chunk(df: pd.DataFrame, source: str) -> pd.DataFrame:
    miss = [c for c in TEST_REQ_MIN_COLS if c not in df.columns]
    if miss:
        raise KeyError(f"{source}: test chunk missing columns: {miss}")

    keep = [c for c in ["row_id", "model_id", "event_id", "node_type", "node_id", "timestep", "water_level_pred"] if c in df.columns]
    out = df[keep]

    if TEST_FILTER_FINITE and "water_level_pred" in out.columns:
        pred = out["water_level_pred"].to_numpy(dtype=np.float32, copy=False)
        keep_mask = np.isfinite(pred)
        if not np.all(keep_mask):
            out = out.loc[keep_mask]

    for c, dt in _dtype_map_for_columns(out.columns.tolist()).items():
        if c in out.columns:
            try:
                out[c] = out[c].astype(dt, copy=False)
            except Exception:
                pass

    return out


def iter_test_chunks(path: Path) -> Iterable[pd.DataFrame]:
    usecols = [c for c in ["row_id", "model_id", "event_id", "node_type", "node_id", "timestep", "water_level_pred"]]
    suffix = path.suffix.lower()

    if suffix == ".csv":
        header = pd.read_csv(path, nrows=0)
        cols = [c for c in usecols if c in header.columns]
        dtypes = _dtype_map_for_columns(cols)
        reader = pd.read_csv(
            path,
            usecols=cols,
            dtype=dtypes if dtypes else None,
            chunksize=TEST_CSV_CHUNK_ROWS,
            low_memory=False,
        )
        for chunk in reader:
            yield chunk
        return

    if suffix in {".parquet", ".pq"}:
        try:
            import pyarrow.parquet as pq

            pf = pq.ParquetFile(path)
            cols = [c for c in usecols if c in set(pf.schema.names)]
            for batch in pf.iter_batches(columns=cols, batch_size=TEST_PARQUET_BATCH_ROWS):
                yield batch.to_pandas()
            return
        except Exception as exc:
            log(f"warn: test parquet batch read failed for {path.name} ({exc}); fallback to full read")
            yield read_table(path, usecols=usecols, kind="test")
            return

    raise ValueError(f"unsupported test table format: {path}")



def read_oof_for_target(path: Path, target: Tuple[int, int], source: str) -> pd.DataFrame:
    model_id, node_type = int(target[0]), int(target[1])
    usecols = list(OOF_REQ_COLS)
    parts: List[pd.DataFrame] = []
    suffix = path.suffix.lower()

    if suffix == ".csv":
        dtypes = _dtype_map_for_columns(usecols)
        reader = pd.read_csv(
            path,
            usecols=usecols,
            dtype=dtypes if dtypes else None,
            chunksize=OOF_CSV_CHUNK_ROWS,
            low_memory=False,
        )
        for i, chunk in enumerate(reader, 1):
            mid = chunk["model_id"].to_numpy(dtype=np.int16, copy=False)
            nty = chunk["node_type"].to_numpy(dtype=np.int16, copy=False)
            mask = (mid == model_id) & (nty == node_type)
            if np.any(mask):
                parts.append(chunk.loc[mask, usecols].copy())
            if i % 10 == 0:
                log(f"oof read progress source={source} target=({model_id},{node_type}) chunk={i}")

    elif suffix in {".parquet", ".pq"}:
        try:
            import pyarrow.parquet as pq

            pf = pq.ParquetFile(path)
            for i, batch in enumerate(
                pf.iter_batches(columns=usecols, batch_size=OOF_PARQUET_BATCH_ROWS),
                1,
            ):
                chunk = batch.to_pandas()
                mid = chunk["model_id"].to_numpy(dtype=np.int16, copy=False)
                nty = chunk["node_type"].to_numpy(dtype=np.int16, copy=False)
                mask = (mid == model_id) & (nty == node_type)
                if np.any(mask):
                    parts.append(chunk.loc[mask, usecols].copy())
                if i % 10 == 0:
                    log(f"oof parquet progress source={source} target=({model_id},{node_type}) batch={i}")
        except Exception as exc:
            log(f"warn: pyarrow batch read failed for {path.name} ({exc}); fallback to pandas")
            df = read_table(path, usecols=usecols, kind="oof")
            mid = df["model_id"].to_numpy(dtype=np.int16, copy=False)
            nty = df["node_type"].to_numpy(dtype=np.int16, copy=False)
            mask = (mid == model_id) & (nty == node_type)
            if np.any(mask):
                parts.append(df.loc[mask, usecols].copy())
    else:
        raise ValueError(f"unsupported OOF file format: {path}")

    if not parts:
        return pd.DataFrame(columns=OOF_REQ_COLS)

    out = pd.concat(parts, ignore_index=True)
    for c, dt in _dtype_map_for_columns(out.columns.tolist()).items():
        if c in out.columns:
            try:
                out[c] = out[c].astype(dt, copy=False)
            except Exception:
                pass
    return out


# ============================================================
# Closed-form simplex blend
# ============================================================


def project_to_simplex(v: np.ndarray) -> np.ndarray:
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u) - 1.0
    ind = np.arange(1, v.size + 1)
    cond = u - cssv / ind > 0
    if not np.any(cond):
        return np.ones_like(v) / float(v.size)
    rho = int(ind[cond][-1])
    theta = cssv[rho - 1] / float(rho)
    w = np.maximum(v - theta, 0.0)
    s = float(w.sum())
    if s <= 0.0:
        return np.ones_like(v) / float(v.size)
    return w / s



def solve_face_qp(G: np.ndarray, b: np.ndarray, active: List[int]) -> np.ndarray | None:
    k = len(active)
    Gs = G[np.ix_(active, active)]
    bs = b[active]

    A = np.zeros((k + 1, k + 1), dtype=np.float64)
    rhs = np.zeros((k + 1,), dtype=np.float64)
    A[:k, :k] = Gs
    A[:k, k] = 1.0
    A[k, :k] = 1.0
    rhs[:k] = bs
    rhs[k] = 1.0

    try:
        sol = np.linalg.solve(A, rhs)
    except np.linalg.LinAlgError:
        sol = np.linalg.lstsq(A, rhs, rcond=None)[0]

    ws = sol[:k]
    if np.any(ws < -1e-12):
        return None

    w = np.zeros((G.shape[0],), dtype=np.float64)
    w[np.asarray(active, dtype=np.int64)] = np.maximum(ws, 0.0)
    s = float(w.sum())
    if s <= 0.0:
        return None
    return w / s



def solve_simplex_closed_form(P: np.ndarray, y: np.ndarray, max_models_exact: int = 8) -> np.ndarray:
    m = P.shape[1]
    if m == 1:
        return np.array([1.0], dtype=np.float64)

    G = P.T @ P
    b = P.T @ y

    if m <= max_models_exact:
        best_w = None
        best_obj = np.inf
        for mask in range(1, 1 << m):
            active = [i for i in range(m) if (mask >> i) & 1]
            w = solve_face_qp(G, b, active)
            if w is None:
                continue
            obj = float(np.mean((P @ w - y) ** 2))
            if obj < best_obj:
                best_obj = obj
                best_w = w
        if best_w is not None:
            return best_w

    # fallback PGD
    w = np.ones((m,), dtype=np.float64) / float(m)
    L = np.linalg.norm(G, ord=2)
    step = 1.0 / float(L + 1e-12)
    for _ in range(500):
        grad = G @ w - b
        w_next = project_to_simplex(w - step * grad)
        if np.max(np.abs(w_next - w)) < 1e-12:
            w = w_next
            break
        w = w_next
    return w


# ============================================================
# Scoring
# ============================================================


def compute_std_rmse_summary(df: pd.DataFrame, pred_col: str, source_name: str) -> pd.DataFrame:
    rec = []
    if df.empty:
        return pd.DataFrame(columns=["source_name", "model_id", "node_type", "num_nodes", "std_rmse"])

    use = df[["model_id", "node_type", "node_id", "event_id", "water_level_true", pred_col]].copy()
    use = use[np.isfinite(use["water_level_true"].to_numpy(dtype=np.float64, copy=False))]
    use = use[np.isfinite(use[pred_col].to_numpy(dtype=np.float64, copy=False))]
    if use.empty:
        return pd.DataFrame(columns=["source_name", "model_id", "node_type", "num_nodes", "std_rmse"])

    for (model_id, node_type, node_id), sub in use.groupby(["model_id", "node_type", "node_id"]):
        event_rmses = []
        for _, ev_sub in sub.groupby("event_id", sort=False):
            diff = ev_sub[pred_col].to_numpy(dtype=np.float64, copy=False) - ev_sub["water_level_true"].to_numpy(dtype=np.float64, copy=False)
            event_rmses.append(float(np.sqrt(np.mean(diff * diff))))
        if not event_rmses:
            continue
        std = float(STD_DEV.get((int(model_id), int(node_type)), 1.0))
        rec.append(
            {
                "source_name": source_name,
                "model_id": int(model_id),
                "node_type": int(node_type),
                "node_id": int(node_id),
                "std_rmse_node": float(np.mean(event_rmses)) / std,
            }
        )

    node_df = pd.DataFrame(rec)
    out = (
        node_df.groupby(["source_name", "model_id", "node_type"], as_index=False)
        .agg(num_nodes=("node_id", "count"), std_rmse=("std_rmse_node", "mean"))
    )
    return out


def build_event_balanced_linear_system(
    train_df: pd.DataFrame,
    pred_cols: List[str],
) -> Tuple[np.ndarray, np.ndarray]:
    event_id = train_df["event_id"].to_numpy(dtype=np.int32, copy=False)
    pred = train_df[pred_cols].to_numpy(dtype=np.float64, copy=False)
    y = train_df["water_level_true"].to_numpy(dtype=np.float64, copy=False)

    ev_unique, ev_inv = np.unique(event_id, return_inverse=True)
    ev_counts = np.bincount(ev_inv, minlength=len(ev_unique)).astype(np.float64, copy=False)
    row_weight = 1.0 / np.maximum(ev_counts[ev_inv], 1.0)
    row_scale = np.sqrt(row_weight)

    return pred * row_scale[:, None], y * row_scale


# ============================================================
# Main blend logic
# ============================================================


def resolve_sample_submission(explicit: str) -> Path | None:
    cands = []
    if explicit:
        cands.append(Path(explicit))

    cands.extend(
        [
            Path("/kaggle/input/urban-flood-modelling/sample_submission.csv"),
            Path("/kaggle/input/urban-flood-modeling/sample_submission.csv"),
            Path("/kaggle/input/competitions/urban-flood-modelling/sample_submission.csv"),
            Path("/kaggle/input/competitions/urban-flood-modeling/sample_submission.csv"),
            Path("sample_submission.csv"),
        ]
    )

    kinput = Path("/kaggle/input")
    if kinput.exists():
        for child in sorted(kinput.iterdir()):
            cand = child / "sample_submission.csv"
            if cand.exists():
                cands.append(cand)

    for p in cands:
        if p.exists():
            return p
    return None



def build_nodewise_weights_and_oof_blend(oof_by_source: Dict[str, pd.DataFrame]) -> Tuple[pd.DataFrame, pd.DataFrame, Dict]:
    sources = sorted(oof_by_source.keys())
    if not sources:
        return (
            pd.DataFrame(columns=NODE_GROUP_COLS + ["source_name", "weight"]),
            pd.DataFrame(columns=OOF_REQ_COLS),
            {"blended_nodes": 0, "single_fallback_nodes": 0, "single_source_fastpath_nodes": 0, "empty_nodes": 0},
        )
    if len(sources) == 1:
        source_name = sources[0]
        source_df = oof_by_source[source_name]
        node_groups = (
            source_df[NODE_GROUP_COLS]
            .drop_duplicates()
            .sort_values(NODE_GROUP_COLS)
            .reset_index(drop=True)
        )
        n_nodes_total = len(node_groups)
        log(f"node groups detected: {n_nodes_total} (single-source fastpath: {source_name})")

        weights_long = node_groups.copy()
        weights_long["source_name"] = source_name
        weights_long["weight"] = 1.0

        oof_blend = source_df[OOF_REQ_COLS].copy()
        oof_blend = oof_blend[
            np.isfinite(oof_blend["water_level_pred"].to_numpy(dtype=np.float64, copy=False))
        ].reset_index(drop=True)

        stats = {
            "blended_nodes": 0,
            "single_fallback_nodes": 0,
            "single_source_fastpath_nodes": int(n_nodes_total),
            "empty_nodes": 0,
        }
        return weights_long, oof_blend, stats

    # group index map for fast node lookup
    group_map: Dict[str, Dict[Tuple[int, int, int], np.ndarray]] = {}
    node_keys_set = set()
    for s in sources:
        g = {}
        grp = oof_by_source[s].groupby(NODE_GROUP_COLS).groups
        for k, idx in grp.items():
            key = (int(k[0]), int(k[1]), int(k[2]))
            g[key] = idx
            node_keys_set.add(key)
        group_map[s] = g

    node_keys = sorted(node_keys_set)
    n_nodes_total = len(node_keys)
    log(f"node groups detected: {n_nodes_total}")

    weight_records = []
    blend_rows = []
    stats = {"blended_nodes": 0, "single_fallback_nodes": 0, "single_source_fastpath_nodes": 0, "empty_nodes": 0}

    for i, node_key in enumerate(node_keys, 1):
        # candidate sources for this node
        cand_sources = [s for s in sources if node_key in group_map[s]]
        if not cand_sources:
            stats["empty_nodes"] += 1
            continue

        merged = None
        pred_cols_all = []
        for s in cand_sources:
            idx = group_map[s][node_key]
            part = oof_by_source[s].loc[idx, OOF_KEY_COLS + ["water_level_true", "water_level_pred"]].copy()
            pcol = f"pred__{s}"
            tcol = f"true__{s}"
            part = part.rename(columns={"water_level_pred": pcol, "water_level_true": tcol})
            pred_cols_all.append(pcol)

            if merged is None:
                merged = part.rename(columns={tcol: "water_level_true"})
            else:
                merged = merged.merge(part, on=OOF_KEY_COLS, how="outer")
                merged["water_level_true"] = merged["water_level_true"].where(
                    merged["water_level_true"].notna(),
                    merged[tcol],
                )
                merged = merged.drop(columns=[tcol], errors="ignore")

        if merged is None or merged.empty:
            stats["empty_nodes"] += 1
            continue

        # choose active sources by coverage (drop sparsest until enough complete rows)
        active_sources = cand_sources.copy()
        while len(active_sources) >= 2:
            active_pred_cols = [f"pred__{s}" for s in active_sources]
            complete = merged[["water_level_true"] + active_pred_cols].dropna()
            if len(complete) >= MIN_OOF_ROWS_NODE:
                break
            counts = {
                s: int(merged[f"pred__{s}"].notna().sum())
                for s in active_sources
            }
            drop_src = min(active_sources, key=lambda s: counts[s])
            active_sources.remove(drop_src)

        weights = {s: 0.0 for s in cand_sources}
        active_pred_cols = [f"pred__{s}" for s in active_sources]
        train_df = merged[["event_id", "water_level_true"] + active_pred_cols].dropna()

        if len(active_sources) >= 2 and len(train_df) >= MIN_OOF_ROWS_NODE:
            P, y = build_event_balanced_linear_system(train_df, active_pred_cols)
            w = solve_simplex_closed_form(P, y, max_models_exact=MAX_MODELS_EXACT)
            for s, wi in zip(active_sources, w.tolist()):
                weights[s] = float(wi)
            stats["blended_nodes"] += 1
        else:
            # single best fallback by node-wise event-average OOF RMSE
            best_src = None
            best_score = np.inf
            for s in cand_sources:
                pcol = f"pred__{s}"
                sub = merged[["event_id", "water_level_true", pcol]].dropna()
                if len(sub) == 0:
                    continue
                event_rmses = []
                for _, ev_sub in sub.groupby("event_id", sort=False):
                    d = ev_sub[pcol].to_numpy(dtype=np.float64, copy=False) - ev_sub["water_level_true"].to_numpy(dtype=np.float64, copy=False)
                    event_rmses.append(float(np.sqrt(np.mean(d * d))))
                if not event_rmses:
                    continue
                score = float(np.mean(event_rmses))
                if score < best_score:
                    best_score = score
                    best_src = s
            if best_src is not None:
                weights[best_src] = 1.0
                stats["single_fallback_nodes"] += 1
            else:
                stats["empty_nodes"] += 1
                continue

        # blended OOF (row-wise normalized by available predictions)
        pred_cols_node = [f"pred__{s}" for s in cand_sources]
        pred_mat = merged[pred_cols_node].to_numpy(dtype=np.float64, copy=False)
        w_vec = np.asarray([weights[s] for s in cand_sources], dtype=np.float64)

        valid = np.isfinite(pred_mat)
        weighted = np.where(valid, pred_mat * w_vec[None, :], 0.0)
        w_eff = np.where(valid, w_vec[None, :], 0.0)
        num = np.sum(weighted, axis=1)
        den = np.sum(w_eff, axis=1)
        blend = np.where(den > 0.0, num / den, np.nan)

        bdf = merged[OOF_KEY_COLS + ["water_level_true"]].copy()
        bdf["water_level_pred"] = blend.astype(np.float32, copy=False)
        bdf = bdf[np.isfinite(bdf["water_level_pred"].to_numpy(dtype=np.float64, copy=False))]
        if not bdf.empty:
            blend_rows.append(bdf)

        model_id, node_type, node_id = node_key
        for s in cand_sources:
            weight_records.append(
                {
                    "model_id": int(model_id),
                    "node_type": int(node_type),
                    "node_id": int(node_id),
                    "source_name": s,
                    "weight": float(weights[s]),
                }
            )

        if i % max(1, LOG_EVERY_NODES) == 0 or i == n_nodes_total:
            log(
                f"node weight progress: {i}/{n_nodes_total} "
                f"blended={stats['blended_nodes']} single={stats['single_fallback_nodes']}"
            )

    weights_long = pd.DataFrame(weight_records)
    oof_blend = pd.concat(blend_rows, ignore_index=True) if blend_rows else pd.DataFrame(columns=OOF_KEY_COLS + ["water_level_true", "water_level_pred"])
    return weights_long, oof_blend, stats



def merge_test_prediction_into_sample(base: pd.DataFrame, test_df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    pred_col = f"pred__{source_name}"
    if "water_level_pred" not in test_df.columns:
        raise RuntimeError(f"{source_name}: test table missing water_level_pred")

    # Priority 1: row_id align (prefer zero-copy fast path)
    if "row_id" in base.columns and "row_id" in test_df.columns:
        b_id = base["row_id"].to_numpy(copy=False)
        t_id = test_df["row_id"].to_numpy(copy=False)

        if len(b_id) == len(t_id) and np.array_equal(b_id, t_id):
            base[pred_col] = test_df["water_level_pred"].to_numpy(dtype=np.float32, copy=False)
            return base

        pred_by_row = pd.Series(
            test_df["water_level_pred"].to_numpy(dtype=np.float32, copy=False),
            index=test_df["row_id"].to_numpy(copy=False),
        )
        base[pred_col] = base["row_id"].map(pred_by_row).to_numpy(dtype=np.float32, copy=False)
        return base

    # Priority 2: key + timestep
    key_t = [c for c in ["model_id", "event_id", "node_type", "node_id", "timestep"] if c in base.columns and c in test_df.columns]
    if len(key_t) == 5:
        b = base[key_t].copy()
        b["__ord"] = np.arange(len(base), dtype=np.int64)
        t = test_df[key_t + ["water_level_pred"]]
        m = b.merge(t, on=key_t, how="left", sort=False).sort_values("__ord")
        base[pred_col] = m["water_level_pred"].to_numpy(dtype=np.float32, copy=False)
        return base

    # Priority 3: key + step_idx
    if all(c in base.columns and c in test_df.columns for c in NODE_KEY_COLS):
        b = base[NODE_KEY_COLS].copy()
        b["__ord"] = np.arange(len(base), dtype=np.int64)
        if "timestep" in base.columns:
            b["__step_idx"] = base["timestep"].to_numpy(dtype=np.int32, copy=False)
        else:
            # Keep old behavior semantics (original base row order), but make it deterministic.
            b["__row_ord"] = b["__ord"].to_numpy(dtype=np.int64, copy=False)
            b = b.sort_values(NODE_KEY_COLS + ["__row_ord", "__ord"], kind="mergesort")
            b["__step_idx"] = b.groupby(NODE_KEY_COLS, sort=False).cumcount().astype(np.int32, copy=False)
            b = b.sort_values("__ord", kind="mergesort").drop(columns=["__row_ord"])

        t = test_df.copy()
        if "timestep" in t.columns:
            if "timestep" in base.columns:
                t["__step_idx"] = t["timestep"].to_numpy(dtype=np.int32, copy=False)
            else:
                t_min = t.groupby(NODE_KEY_COLS, sort=False)["timestep"].transform("min").to_numpy(dtype=np.int32, copy=False)
                t_step = t["timestep"].to_numpy(dtype=np.int32, copy=False) - t_min
                t["__step_idx"] = t_step.astype(np.int32, copy=False)
                log(f"{source_name}: normalized source timestep to zero-based step index for key+step merge")
        else:
            t["__ord"] = np.arange(len(t), dtype=np.int64)
            sort_cols = NODE_KEY_COLS + (["row_id"] if "row_id" in t.columns else ["__ord"])
            t = t.sort_values(sort_cols, kind="mergesort")
            t["__step_idx"] = t.groupby(NODE_KEY_COLS, sort=False).cumcount().astype(np.int32, copy=False)

        m = b.merge(
            t[NODE_KEY_COLS + ["__step_idx", "water_level_pred"]],
            on=NODE_KEY_COLS + ["__step_idx"],
            how="left",
            sort=False,
        ).sort_values("__ord")
        base[pred_col] = m["water_level_pred"].to_numpy(dtype=np.float32, copy=False)
        return base

    raise RuntimeError(
        f"{source_name}: cannot align test prediction to sample (needs row_id or key columns)"
    )


def accumulate_source_stream_rowid(
    source_name: str,
    src_obj: object,
    ws: pd.DataFrame | None,
    use_weight: bool,
    n_rows: int,
) -> Dict[str, object]:
    """Stream one source by row_id and return local accumulators."""
    if isinstance(src_obj, pd.DataFrame):
        raise RuntimeError(f"{source_name}: DataFrame input is not supported in parallel stream path")

    test_path = Path(src_obj)
    equal_num = np.zeros((n_rows,), dtype=np.float32)
    equal_cnt = np.zeros((n_rows,), dtype=np.uint16)
    weighted_num = np.zeros((n_rows,), dtype=np.float32) if use_weight else None
    weighted_den = np.zeros((n_rows,), dtype=np.float32) if use_weight else None

    source_rows = 0
    valid_total = 0

    target_pairs = None
    if TEST_FILTER_BY_SOURCE_TARGETS and use_weight and ws is not None and len(ws) > 0:
        target_pairs = sorted(
            set(
                (int(r.model_id), int(r.node_type))
                for r in ws[["model_id", "node_type"]].itertuples(index=False)
            )
        )

    for c_idx, chunk in enumerate(iter_test_chunks(test_path), 1):
        source_rows += int(len(chunk))
        if len(chunk) == 0:
            continue

        chunk = normalize_test_chunk(chunk, source_name)
        if len(chunk) == 0:
            continue

        if target_pairs is not None and {"model_id", "node_type"}.issubset(chunk.columns):
            mid = chunk["model_id"].to_numpy(dtype=np.int16, copy=False)
            nty = chunk["node_type"].to_numpy(dtype=np.int16, copy=False)
            pair_mask = np.zeros((len(chunk),), dtype=bool)
            for pm, pn in target_pairs:
                pair_mask |= ((mid == pm) & (nty == pn))
            if not np.any(pair_mask):
                if c_idx % SUBMISSION_LOG_EVERY_CHUNKS == 0:
                    log(
                        f"parallel test progress source={source_name} "
                        f"chunk={c_idx} rows_seen={source_rows}"
                    )
                continue
            chunk = chunk.loc[pair_mask]
            if len(chunk) == 0:
                continue

        if "row_id" not in chunk.columns:
            raise RuntimeError(f"{source_name}: row_id missing in chunk for streaming path")

        rid = chunk["row_id"].to_numpy(dtype=np.int64, copy=False)
        pred = chunk["water_level_pred"].to_numpy(dtype=np.float32, copy=False)
        in_range = (rid >= 0) & (rid < n_rows)
        finite = np.isfinite(pred)
        use_eq = in_range & finite
        if not np.any(use_eq):
            if c_idx % SUBMISSION_LOG_EVERY_CHUNKS == 0:
                log(
                    f"parallel test progress source={source_name} "
                    f"chunk={c_idx} rows_seen={source_rows}"
                )
            continue

        idx_eq = rid[use_eq].astype(np.int64, copy=False)
        pred_eq = pred[use_eq]
        np.add.at(equal_num, idx_eq, pred_eq)
        np.add.at(equal_cnt, idx_eq, 1)
        valid_total += int(use_eq.sum())

        if use_weight and ws is not None and set(NODE_GROUP_COLS).issubset(chunk.columns):
            sub = chunk.loc[use_eq, NODE_GROUP_COLS].copy()
            wvals = (
                sub.merge(ws, on=NODE_GROUP_COLS, how="left", sort=False)["weight"]
                .to_numpy(dtype=np.float32, copy=False)
            )
            wvals = np.nan_to_num(wvals, nan=0.0, posinf=0.0, neginf=0.0)
            use_w = wvals > 0.0
            if np.any(use_w):
                idx_w = idx_eq[use_w]
                pred_w = pred_eq[use_w]
                w_w = wvals[use_w]
                np.add.at(weighted_num, idx_w, pred_w * w_w)
                np.add.at(weighted_den, idx_w, w_w)

        if c_idx % SUBMISSION_LOG_EVERY_CHUNKS == 0:
            log(
                f"parallel test progress source={source_name} chunk={c_idx} "
                f"rows_seen={source_rows} valid_rows={valid_total}"
            )

    return {
        "source_name": source_name,
        "source_rows": int(source_rows),
        "valid_total": int(valid_total),
        "equal_num": equal_num,
        "equal_cnt": equal_cnt,
        "weighted_num": weighted_num,
        "weighted_den": weighted_den,
    }


def build_submission_from_weights(
    sample_path: Path,
    test_inputs_by_source: Dict[str, object],
    weights_long: pd.DataFrame,
) -> Tuple[pd.DataFrame, Dict]:
    sample = pd.read_csv(sample_path)
    n_rows = len(sample)

    if n_rows == 0:
        raise RuntimeError("sample_submission is empty")

    have_node_cols = all(c in sample.columns for c in NODE_GROUP_COLS)

    # Keep memory bounded: accumulate blend statistics incrementally.
    weighted_num = np.zeros((n_rows,), dtype=np.float32)
    weighted_den = np.zeros((n_rows,), dtype=np.float32)
    equal_num = np.zeros((n_rows,), dtype=np.float32)
    equal_cnt = np.zeros((n_rows,), dtype=np.uint16)

    weights_by_source: Dict[str, pd.DataFrame] = {}
    if not weights_long.empty:
        wl = weights_long[["source_name"] + NODE_GROUP_COLS + ["weight"]].copy()
        wl = wl.drop_duplicates(["source_name"] + NODE_GROUP_COLS, keep="last")
        for s_name, g in wl.groupby("source_name", sort=False):
            weights_by_source[s_name] = g[NODE_GROUP_COLS + ["weight"]].copy()

    used_sources = []
    source_names = sorted(test_inputs_by_source.keys())
    n_sources = len(source_names)
    sample_rowid_identity = False
    if "row_id" in sample.columns:
        row_id = sample["row_id"]
        try:
            first = int(row_id.iloc[0]) if len(row_id) > 0 else 0
            last = int(row_id.iloc[-1]) if len(row_id) > 0 else -1
            sample_rowid_identity = (
                bool(row_id.is_monotonic_increasing)
                and bool(row_id.is_unique)
                and first == 0
                and last == (n_rows - 1)
                and (last - first + 1 == n_rows)
            )
        except Exception:
            sample_rowid_identity = False

    # Parallel pass (stream+row_id fastpath only): source-local accumulation then reduce.
    processed_sources = set()
    if TEST_SOURCE_PARALLEL > 1 and TEST_STREAM_ROWID_FASTPATH and sample_rowid_identity:
        parallel_jobs = []
        for s in source_names:
            src_obj = test_inputs_by_source[s]
            if isinstance(src_obj, pd.DataFrame):
                continue
            ws = weights_by_source.get(s)
            use_weight = have_node_cols and ws is not None and len(ws) > 0
            parallel_jobs.append((s, src_obj, ws, use_weight))

        if parallel_jobs:
            n_workers = min(TEST_SOURCE_PARALLEL, len(parallel_jobs))
            log(
                f"parallel source stream: workers={n_workers} "
                f"jobs={len(parallel_jobs)} (row_id fastpath only)"
            )
            with ThreadPoolExecutor(max_workers=n_workers) as ex:
                fut_map = {
                    ex.submit(
                        accumulate_source_stream_rowid,
                        s,
                        src_obj,
                        ws,
                        use_weight,
                        n_rows,
                    ): (s, use_weight)
                    for (s, src_obj, ws, use_weight) in parallel_jobs
                }
                for fut in as_completed(fut_map):
                    s, use_weight = fut_map[fut]
                    try:
                        res = fut.result()
                    except Exception as exc:
                        log(f"parallel stream fallback: source={s} reason={exc}")
                        continue

                    equal_num += res["equal_num"]
                    equal_cnt += res["equal_cnt"]
                    if use_weight and res["weighted_num"] is not None:
                        weighted_num += res["weighted_num"]
                        weighted_den += res["weighted_den"]
                    used_sources.append(s)
                    processed_sources.add(s)
                    log(
                        f"test merge(parallel): source={s} rows={res['source_rows']} "
                        f"valid_pred_rows={res['valid_total']}"
                    )
                    del res
                    gc.collect()

    for s_idx, s in enumerate(source_names, 1):
        if s in processed_sources:
            continue
        src_obj = test_inputs_by_source[s]
        ws = weights_by_source.get(s)
        use_weight = have_node_cols and ws is not None and len(ws) > 0
        streamed_ok = False
        valid_total = 0
        source_rows = 0

        # Streaming fast path: row_id-indexed accumulation without loading full test table.
        if (
            TEST_STREAM_ROWID_FASTPATH
            and sample_rowid_identity
            and not isinstance(src_obj, pd.DataFrame)
        ):
            try:
                test_path = Path(src_obj)
                target_pairs = None
                if TEST_FILTER_BY_SOURCE_TARGETS and use_weight:
                    target_pairs = sorted(
                        set(
                            (int(r.model_id), int(r.node_type))
                            for r in ws[["model_id", "node_type"]].itertuples(index=False)
                        )
                    )

                for c_idx, chunk in enumerate(iter_test_chunks(test_path), 1):
                    source_rows += int(len(chunk))
                    if len(chunk) == 0:
                        continue

                    chunk = normalize_test_chunk(chunk, s)
                    if len(chunk) == 0:
                        continue

                    if target_pairs is not None and {"model_id", "node_type"}.issubset(chunk.columns):
                        mid = chunk["model_id"].to_numpy(dtype=np.int16, copy=False)
                        nty = chunk["node_type"].to_numpy(dtype=np.int16, copy=False)
                        pair_mask = np.zeros((len(chunk),), dtype=bool)
                        for pm, pn in target_pairs:
                            pair_mask |= ((mid == pm) & (nty == pn))
                        if not np.any(pair_mask):
                            if c_idx % SUBMISSION_LOG_EVERY_CHUNKS == 0:
                                log(f"test blend progress source={s} chunk={c_idx} rows_seen={source_rows}")
                            continue
                        chunk = chunk.loc[pair_mask]
                        if len(chunk) == 0:
                            continue

                    if "row_id" not in chunk.columns:
                        raise RuntimeError(f"{s}: row_id missing in chunk for streaming path")

                    rid = chunk["row_id"].to_numpy(dtype=np.int64, copy=False)
                    pred = chunk["water_level_pred"].to_numpy(dtype=np.float32, copy=False)
                    in_range = (rid >= 0) & (rid < n_rows)
                    finite = np.isfinite(pred)
                    use_eq = in_range & finite
                    if not np.any(use_eq):
                        if c_idx % SUBMISSION_LOG_EVERY_CHUNKS == 0:
                            log(f"test blend progress source={s} chunk={c_idx} rows_seen={source_rows}")
                        continue

                    idx_eq = rid[use_eq].astype(np.int64, copy=False)
                    pred_eq = pred[use_eq]
                    np.add.at(equal_num, idx_eq, pred_eq)
                    np.add.at(equal_cnt, idx_eq, 1)
                    valid_total += int(use_eq.sum())

                    if use_weight and set(NODE_GROUP_COLS).issubset(chunk.columns):
                        sub = chunk.loc[use_eq, NODE_GROUP_COLS].copy()
                        wvals = (
                            sub.merge(ws, on=NODE_GROUP_COLS, how="left", sort=False)["weight"]
                            .to_numpy(dtype=np.float32, copy=False)
                        )
                        wvals = np.nan_to_num(wvals, nan=0.0, posinf=0.0, neginf=0.0)
                        use_w = wvals > 0.0
                        if np.any(use_w):
                            idx_w = idx_eq[use_w]
                            pred_w = pred_eq[use_w]
                            w_w = wvals[use_w]
                            np.add.at(weighted_num, idx_w, pred_w * w_w)
                            np.add.at(weighted_den, idx_w, w_w)

                    if c_idx % SUBMISSION_LOG_EVERY_CHUNKS == 0:
                        log(
                            f"test blend progress source={s} chunk={c_idx} "
                            f"rows_seen={source_rows} valid_rows={valid_total}"
                        )

                streamed_ok = True
            except Exception as exc:
                log(f"test stream fallback: source={s} reason={exc}")

        if not streamed_ok:
            test_df = None
            try:
                if isinstance(src_obj, pd.DataFrame):
                    test_df = src_obj
                else:
                    test_path = Path(src_obj)
                    test_df = normalize_test(read_table(test_path, kind="test"), s)

                sample = merge_test_prediction_into_sample(sample, test_df, s)
            except Exception as exc:
                log(f"test merge skipped: source={s} reason={exc}")
                continue

            pred_col = f"pred__{s}"
            if pred_col not in sample.columns:
                log(f"test merge skipped: source={s} reason=missing {pred_col}")
                continue

            n_chunks = (n_rows + SUBMISSION_BLEND_CHUNK_ROWS - 1) // SUBMISSION_BLEND_CHUNK_ROWS
            for c_idx, st in enumerate(range(0, n_rows, SUBMISSION_BLEND_CHUNK_ROWS), 1):
                ed = min(n_rows, st + SUBMISSION_BLEND_CHUNK_ROWS)
                sl = slice(st, ed)

                pred_chunk = sample[pred_col].iloc[sl].to_numpy(dtype=np.float32, copy=False)
                valid = np.isfinite(pred_chunk)
                if not np.any(valid):
                    continue

                valid_total += int(valid.sum())
                eq_num_sl = equal_num[sl]
                eq_cnt_sl = equal_cnt[sl]
                eq_num_sl[valid] += pred_chunk[valid]
                eq_cnt_sl[valid] += 1

                if use_weight:
                    w_chunk = (
                        sample.iloc[sl][NODE_GROUP_COLS]
                        .merge(ws, on=NODE_GROUP_COLS, how="left", sort=False)["weight"]
                        .to_numpy(dtype=np.float32, copy=False)
                    )
                    w_chunk = np.nan_to_num(w_chunk, nan=0.0, posinf=0.0, neginf=0.0)
                    use = valid & (w_chunk > 0.0)
                    if np.any(use):
                        wn_sl = weighted_num[sl]
                        wd_sl = weighted_den[sl]
                        wn_sl[use] += pred_chunk[use] * w_chunk[use]
                        wd_sl[use] += w_chunk[use]

                if c_idx % SUBMISSION_LOG_EVERY_CHUNKS == 0 or c_idx == n_chunks:
                    log(
                        f"test blend progress source={s} {c_idx}/{n_chunks} "
                        f"rows={ed}/{n_rows}"
                    )

            source_rows = int(len(test_df))
            sample.drop(columns=[pred_col], inplace=True, errors="ignore")
            test_df = None
            gc.collect()

        used_sources.append(s)
        log(
            f"test merge: source={s} rows={source_rows} "
            f"valid_pred_rows={valid_total} source_index={s_idx}/{n_sources}"
        )
        gc.collect()

    if not used_sources:
        raise RuntimeError("no usable test prediction sources after alignment")

    learned_mask = weighted_den > 0.0
    blend_learned = np.full((n_rows,), np.nan, dtype=np.float32)
    blend_learned[learned_mask] = weighted_num[learned_mask] / weighted_den[learned_mask]

    equal_mask = equal_cnt > 0
    blend_equal = np.full((n_rows,), np.nan, dtype=np.float32)
    blend_equal[equal_mask] = equal_num[equal_mask] / equal_cnt[equal_mask].astype(np.float32)

    blend = np.where(np.isfinite(blend_learned), blend_learned, blend_equal)

    if "water_level" in sample.columns:
        base_wl = sample["water_level"].to_numpy(dtype=np.float32, copy=False)
        miss = ~np.isfinite(blend)
        if np.any(miss):
            blend[miss] = base_wl[miss]
    else:
        global_mean = float(np.nanmean(blend)) if np.isfinite(blend).any() else 0.0
        blend = np.where(np.isfinite(blend), blend, np.float32(global_mean))

    sample["water_level"] = blend.astype(np.float32, copy=False)

    coverage = {
        "rows_total": int(n_rows),
        "rows_with_learned_weights": int(np.sum(learned_mask)),
        "rows_with_equal_fallback": int(np.sum((~learned_mask) & equal_mask)),
        "rows_with_base_fill": int(np.sum(~equal_mask)),
        "used_sources": used_sources,
    }
    return sample, coverage



# ============================================================
# Run
# ============================================================
# ============================================================
# Run
# ============================================================

def main() -> None:
    t0 = time.time()
    log("Step 0: discover bundles")

    roots = [p.resolve() for p in BUNDLE_ROOTS]
    log(f"bundle roots={roots}")
    log("objective_mode=event_balanced_weighted_sse")
    log(
        "TABLE_OPTIONS="
        f"defer_test_load={DEFER_TEST_LOAD} sort_normalized={SORT_NORMALIZED_TABLES} "
        f"strict_checks={STRICT_TABLE_CHECKS} compute_node_groups={COMPUTE_SOURCE_NODE_GROUPS}"
    )
    log(
        "TARGET_OPTIONS="
        f"targets={TARGETS} oof_csv_chunk_rows={OOF_CSV_CHUNK_ROWS} "
        f"oof_parquet_batch_rows={OOF_PARQUET_BATCH_ROWS} "
        f"include_oof_empty_in_test={INCLUDE_OOF_EMPTY_SOURCES_IN_TEST_BLEND}"
    )
    log(
        "TEST_OPTIONS="
        f"stream_rowid_fastpath={TEST_STREAM_ROWID_FASTPATH} "
        f"filter_by_source_targets={TEST_FILTER_BY_SOURCE_TARGETS} "
        f"test_source_parallel={TEST_SOURCE_PARALLEL} "
        f"test_csv_chunk_rows={TEST_CSV_CHUNK_ROWS} "
        f"test_parquet_batch_rows={TEST_PARQUET_BATCH_ROWS} "
        f"submission_chunk_rows={SUBMISSION_BLEND_CHUNK_ROWS}"
    )

    manifest_paths = discover_manifest_paths(roots, MANIFEST_NAME)
    bundles: List[Bundle] = []

    for mp in manifest_paths:
        try:
            bundles.append(load_manifest_bundle(mp))
        except Exception as exc:
            log(f"manifest skipped: {mp} reason={exc}")

    if not bundles and not STRICT_REQUIRE_MANIFESTS:
        log("no valid manifests; fallback to loose oof/test pair detection")
        bundles.extend(discover_loose_bundles(roots))

    if not bundles:
        raise RuntimeError("no usable bundles found")

    bundles = uniquify_bundle_names(bundles)

    mismatched = [
        b for b in bundles
        if b.n_folds > 0 and (b.n_folds != EXPECTED_FOLDS or b.event_split_seed != EXPECTED_SEED)
    ]
    if mismatched and not ALLOW_FOLD_MISMATCH:
        raise RuntimeError(
            "fold mismatch bundles detected: "
            + ", ".join(f"{b.name}(n_folds={b.n_folds},seed={b.event_split_seed})" for b in mismatched)
        )
    if mismatched:
        log(
            "fold mismatch allowed: "
            + ", ".join(f"{b.name}(n_folds={b.n_folds},seed={b.event_split_seed})" for b in mismatched)
        )

    log(f"detected bundles={len(bundles)}")
    bundle_by_name: Dict[str, Bundle] = {}
    source_stats = {}
    for b in bundles:
        bundle_by_name[b.name] = b
        source_stats[b.name] = {
            "source_name": b.name,
            "bundle_name": b.bundle_name,
            "source_kind": b.source_kind,
            "n_folds": b.n_folds,
            "event_split_seed": b.event_split_seed,
            "oof_rows": 0,
            "test_rows": np.nan,
            "num_node_groups": 0,
            "targets": set(),
            "oof_path": str(b.oof_path),
            "test_path": str(b.test_path),
            "manifest_path": str(b.manifest_path) if b.manifest_path is not None else "",
        }
        log(
            f"bundle={b.name} original={b.bundle_name} kind={b.source_kind} n_folds={b.n_folds} seed={b.event_split_seed} "
            f"oof={b.oof_path} test={b.test_path}"
        )

    log("Step 1: target-wise OOF load + closed-form solve")
    weights_parts: List[pd.DataFrame] = []
    oof_blend_parts: List[pd.DataFrame] = []
    cv_frames: List[pd.DataFrame] = []
    blend_stats_by_target = {}

    for target in TARGETS:
        model_id, node_type = int(target[0]), int(target[1])
        log(f"target=({model_id},{node_type}) load OOF from sources")

        oof_by_source: Dict[str, pd.DataFrame] = {}
        for b in bundles:
            try:
                oof_raw = read_oof_for_target(b.oof_path, (model_id, node_type), b.name)
                if len(oof_raw) == 0:
                    continue
                oof = normalize_oof(oof_raw, b.name)
            except Exception as exc:
                log(f"target=({model_id},{node_type}) source skipped: {b.name} reason={exc}")
                continue

            if len(oof) == 0:
                continue

            oof_by_source[b.name] = oof
            st = source_stats[b.name]
            st["oof_rows"] += int(len(oof))
            st["targets"].add(f"{model_id}:{node_type}")
            if COMPUTE_SOURCE_NODE_GROUPS:
                st["num_node_groups"] += int(oof[NODE_GROUP_COLS].drop_duplicates().shape[0])

            ng_log = int(oof[NODE_GROUP_COLS].drop_duplicates().shape[0]) if COMPUTE_SOURCE_NODE_GROUPS else "skip"
            log(
                f"target=({model_id},{node_type}) loaded source={b.name} "
                f"oof_rows={len(oof)} node_groups={ng_log}"
            )

        if len(oof_by_source) < 1:
            log(f"target=({model_id},{node_type}) skipped: no OOF rows from any source")
            continue

        weights_t, oof_blend_t, stats_t = build_nodewise_weights_and_oof_blend(oof_by_source)
        blend_stats_by_target[f"{model_id}:{node_type}"] = stats_t
        log(
            f"target=({model_id},{node_type}) solve done: blended_nodes={stats_t['blended_nodes']} "
            f"single_fallback_nodes={stats_t['single_fallback_nodes']} "
            f"single_source_fastpath_nodes={stats_t.get('single_source_fastpath_nodes', 0)} "
            f"empty_nodes={stats_t['empty_nodes']}"
        )

        if not weights_t.empty:
            weights_t = weights_t[
                (weights_t["model_id"] == model_id) & (weights_t["node_type"] == node_type)
            ].reset_index(drop=True)
            if len(weights_t) > 0:
                weights_parts.append(weights_t)
                w_mean = (
                    weights_t.groupby("source_name", as_index=False)["weight"].mean()
                    .sort_values("weight", ascending=False)
                )
                log(
                    f"target=({model_id},{node_type}) mean_weights="
                    + ", ".join(f"{r.source_name}:{float(r.weight):.4f}" for r in w_mean.itertuples(index=False))
                )

        if SAVE_BLEND_OOF and not oof_blend_t.empty:
            oof_blend_t = oof_blend_t[
                (oof_blend_t["model_id"] == model_id) & (oof_blend_t["node_type"] == node_type)
            ].reset_index(drop=True)
            if len(oof_blend_t) > 0:
                oof_blend_parts.append(oof_blend_t)

        for s_name, oof_df in sorted(oof_by_source.items()):
            score_df = compute_std_rmse_summary(oof_df, "water_level_pred", s_name)
            if score_df.empty:
                continue
            score_df = score_df[
                (score_df["model_id"] == model_id) & (score_df["node_type"] == node_type)
            ]
            if score_df.empty:
                continue
            cv_frames.append(score_df)
            score = float(score_df["std_rmse"].iloc[0])
            log(f"target=({model_id},{node_type}) cv source={s_name} std_rmse={score:.6f}")

        if not oof_blend_t.empty:
            blend_score_df = compute_std_rmse_summary(oof_blend_t, "water_level_pred", "blend")
            blend_score_df = blend_score_df[
                (blend_score_df["model_id"] == model_id) & (blend_score_df["node_type"] == node_type)
            ]
            if not blend_score_df.empty:
                cv_frames.append(blend_score_df)
                blend_score = float(blend_score_df["std_rmse"].iloc[0])
                log(f"target=({model_id},{node_type}) cv source=blend std_rmse={blend_score:.6f}")

        oof_by_source.clear()
        gc.collect()

    if not weights_parts:
        raise RuntimeError("no target produced node weights (all OOF empty?)")

    weights_long = pd.concat(weights_parts, ignore_index=True)
    weights_long = weights_long.drop_duplicates(subset=NODE_GROUP_COLS + ["source_name"], keep="last").reset_index(drop=True)

    if SAVE_BLEND_OOF and oof_blend_parts:
        oof_blend = pd.concat(oof_blend_parts, ignore_index=True)
        oof_blend = oof_blend.drop_duplicates(subset=OOF_KEY_COLS, keep="last").reset_index(drop=True)
    else:
        oof_blend = pd.DataFrame(columns=OOF_KEY_COLS + ["water_level_true", "water_level_pred"])

    cv_summary = pd.concat(cv_frames, ignore_index=True) if cv_frames else pd.DataFrame(
        columns=["source_name", "model_id", "node_type", "num_nodes", "std_rmse"]
    )
    if len(cv_summary) > 0:
        cv_summary = cv_summary.drop_duplicates(subset=["source_name", "model_id", "node_type"], keep="last").reset_index(drop=True)

    used_source_names = sorted(set(weights_long["source_name"].dropna().astype(str).tolist()))
    if INCLUDE_OOF_EMPTY_SOURCES_IN_TEST_BLEND:
        test_source_names = sorted(bundle_by_name.keys())
    else:
        test_source_names = used_source_names

    test_inputs_by_source: Dict[str, object] = {
        s: bundle_by_name[s].test_path for s in test_source_names if s in bundle_by_name
    }

    log("Step 2: build submission from node weights")
    sample_path = resolve_sample_submission(SAMPLE_SUBMISSION_PATH)
    if sample_path is None:
        raise RuntimeError("sample_submission.csv not found; set SAMPLE_SUBMISSION_PATH")
    log(f"sample_submission={sample_path}")
    log(f"test sources for blend={sorted(test_inputs_by_source.keys())}")

    submission, coverage = build_submission_from_weights(sample_path, test_inputs_by_source, weights_long)
    test_inputs_by_source.clear()
    gc.collect()

    log(
        f"submission coverage: total={coverage['rows_total']} "
        f"learned={coverage['rows_with_learned_weights']} "
        f"equal_fallback={coverage['rows_with_equal_fallback']} "
        f"base_fill={coverage['rows_with_base_fill']}"
    )

    source_rows = []
    for s_name in sorted(source_stats.keys()):
        st = source_stats[s_name]
        targets_txt = ",".join(sorted(st["targets"])) if st["targets"] else ""
        num_groups = int(st["num_node_groups"]) if COMPUTE_SOURCE_NODE_GROUPS else -1
        source_rows.append(
            {
                "source_name": st["source_name"],
                "bundle_name": st["bundle_name"],
                "source_kind": st["source_kind"],
                "n_folds": st["n_folds"],
                "event_split_seed": st["event_split_seed"],
                "oof_rows": int(st["oof_rows"]),
                "test_rows": st["test_rows"],
                "num_node_groups": num_groups,
                "targets": targets_txt,
                "oof_path": st["oof_path"],
                "test_path": st["test_path"],
                "manifest_path": st["manifest_path"],
            }
        )

    log("Step 3: save outputs")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    sub_path = OUTPUT_DIR / "submission.csv"
    weights_path = OUTPUT_DIR / "node_weights.csv"
    source_path = OUTPUT_DIR / "detected_sources.csv"
    cv_path = OUTPUT_DIR / "cv_summary.csv"
    meta_path = OUTPUT_DIR / "blend_metadata.json"

    submission.to_csv(sub_path, index=False)
    public_sub_path = None
    if PUBLIC_SUBMISSION_PATH:
        public_sub_path = Path(PUBLIC_SUBMISSION_PATH)
        public_sub_path.parent.mkdir(parents=True, exist_ok=True)
        submission.to_csv(public_sub_path, index=False)
    weights_long.to_csv(weights_path, index=False)
    pd.DataFrame(source_rows).to_csv(source_path, index=False)
    cv_summary.to_csv(cv_path, index=False)

    parquet_ok = True
    try:
        weights_long.to_parquet(OUTPUT_DIR / "node_weights.parquet", index=False)
    except Exception as exc:
        parquet_ok = False
        log(f"warn: parquet write failed for node_weights ({exc.__class__.__name__}: {exc})")

    if SAVE_BLEND_OOF:
        try:
            oof_blend.to_parquet(OUTPUT_DIR / "oof_blend_predictions.parquet", index=False)
        except Exception:
            oof_blend.to_csv(OUTPUT_DIR / "oof_blend_predictions.csv", index=False)

    num_used_sources = len(used_source_names)
    meta = {
        "schema_version": SCHEMA_VERSION,
        "created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "bundle_roots": [str(p) for p in roots],
        "manifest_name": MANIFEST_NAME,
        "num_detected_bundles": len(bundles),
        "num_used_sources": num_used_sources,
        "used_sources": used_source_names,
        "targets": [f"{m}:{t}" for (m, t) in TARGETS],
        "expected_folds": EXPECTED_FOLDS,
        "expected_seed": EXPECTED_SEED,
        "allow_fold_mismatch": ALLOW_FOLD_MISMATCH,
        "max_models_exact": MAX_MODELS_EXACT,
        "min_oof_rows_node": MIN_OOF_ROWS_NODE,
        "objective_mode": "event_balanced_weighted_sse",
        "defer_test_load": DEFER_TEST_LOAD,
        "sort_normalized_tables": SORT_NORMALIZED_TABLES,
        "strict_table_checks": STRICT_TABLE_CHECKS,
        "compute_source_node_groups": COMPUTE_SOURCE_NODE_GROUPS,
        "oof_csv_chunk_rows": OOF_CSV_CHUNK_ROWS,
        "oof_parquet_batch_rows": OOF_PARQUET_BATCH_ROWS,
        "test_source_parallel": TEST_SOURCE_PARALLEL,
        "include_oof_empty_sources_in_test_blend": INCLUDE_OOF_EMPTY_SOURCES_IN_TEST_BLEND,
        "blend_stats_by_target": blend_stats_by_target,
        "coverage": coverage,
        "parquet_weights_written": parquet_ok,
        "output_dir": str(OUTPUT_DIR),
        "public_submission_path": str(public_sub_path) if public_sub_path is not None else None,
        "elapsed_sec": float(time.time() - t0),
    }
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2))

    log(f"saved: {sub_path}")
    if public_sub_path is not None:
        log(f"saved: {public_sub_path}")
    log(f"saved: {weights_path}")
    log(f"saved: {source_path}")
    log(f"saved: {cv_path}")
    log(f"saved: {meta_path}")
    log(f"done elapsed={time.time()-t0:.1f}s")


if __name__ == "__main__":
    main()


[10:15:44] Step 0: discover bundles
[10:15:44] bundle roots=[PosixPath('/kaggle/input'), PosixPath('/kaggle/working/model_bundles'), PosixPath('/kaggle/working/ensemble/artifacts/model_bundles')]
[10:15:44] objective_mode=event_balanced_weighted_sse
[10:15:44] TABLE_OPTIONS=defer_test_load=True sort_normalized=False strict_checks=False compute_node_groups=False
[10:15:44] TARGET_OPTIONS=targets=[(1, 1), (1, 2), (2, 1), (2, 2)] oof_csv_chunk_rows=2000000 oof_parquet_batch_rows=2000000 include_oof_empty_in_test=False
[10:15:44] TEST_OPTIONS=stream_rowid_fastpath=True filter_by_source_targets=True test_source_parallel=4 test_csv_chunk_rows=2000000 test_parquet_batch_rows=2000000 submission_chunk_rows=2000000
[10:15:47] manifest skipped: /kaggle/input/datasets/mtmrs1/pi-gnn-graphattention-rollout-raingate-l10/bundle-20260313T100031Z-3-001/bundle/bundle_manifest.json reason=/kaggle/input/datasets/mtmrs1/pi-gnn-graphattention-rollout-raingate-l10/bundle-20260313T100031Z-3-001/bundle/bundle_m